<a href="https://colab.research.google.com/github/dtamendarov/MDS-EDA_Netflix/blob/main/course_project_mds_eda_netflix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Курсовой проект, MDS

## Проектная группа:
- Даниил Нифанин
- Данил Николаев
- Дмитрий Тамендаров
- Мария Вичентиевич
- Светлана Максимова

## Общая информация

Стек: pandas, numpy, matplotlib, seaborn  
Источники: NetflixShows.xlsx, OMDB API, открытые датасеты Kaggle


## Введение

Netflix, это международный стримминговый сервис фильмов и сериалов. У сервиса более 300 млн зрителей и у каждого есть предпочтения. Одной из задач является оценка успешности шоу и фильмов, которые будут включены в подписку. Для прогноза успешности продукта нужно уметь делать выводы по описанию фильма, шоу. Для этого компания непрерывно собирает обратную связь от пользователей и хранит ингформацию о каждом шоу. В данной работу предпринята попытка предсказать успешность шоу или фильма по открытой информации о 1000 шоу по состоянию на 11.06.2017 - 1000 Netflix Shows.  
**Цель:** По характеристикам шоу/фильма предсказать успешность в регионах мира.  
**Задачи:**
- предобработка основного датасета
- Предложить новые признаки на основе имеющихся
- обогатить датасет с помощью внешних источников
- демографический анализ
- анализ шоу
- анализ количества и качества оценок
- прогноз на основе проведенного анализа

## Описание датасета 1000 NetflixShows

### Описание признаков

- title - название шоу.
- rating - рейтинг шоу. Например: G, PG, TV-14, TV-MA.
- ratingLevel - описание рейтинговой группы и особенностей шоу.
- ratingDescription - рейтинг шоу, закодированный числом.
- release year - год выпуска
user rating score - оценка пользователей.
- user rating size - общий рейтинг пользователей.

### Требования к проекту

В качестве результата выполнения курсового проекта ваша команда должна получить презентацию и защитить ее перед комиссией.

Оформление презентации остаётся полностью на ваше усмотрение, но помните, что результат должен быть релевантен для демонстрации бизнес-заказчику — комиссию, принимающую вашу работу, правильнее всего воспринимать именно в таком качестве. Например, вставлять в презентацию строчки кода или злоупотреблять скринами блокнота не рекомендуется.

С точки зрения концепции выполнения проекта — вам необходимо принять на себя роль аналитиков: провести работу над признаками, исследовать информацию, содержащуюся в датасете,  выявить тенденции, тренды, факты из данных, а также, конечно, презентовать всё это в понятном виде.

Фактически, можно воспринимать этот проект в следующем ключе: к вам пришел некий бизнес-заказчик — например, это может быть непосредственно представитель Netflix, которые хотят улучшить какие-то процессы; или какая-то компания, которая хочет снять какой-то новый проект и/или продать какой-то свой уже готовый продукт Netflix'у; или это может быть некая компания конкурент Netflix'a, которая заинтересована в общем исследовании рынка; или абсолютно любой другой стейхолдер в рамках данной отрасли — и вот этот заказчик просит вас проанализировать данные и извлечь на основе них какие-то полезные и значимые бизнес-инсайты для него. Разумеется, чем глубже, чем осмысленнее и чем нетривиальнее будут эти выводы, тем больше они вам заплатят :)


## Предобработка данных

### Импортирование датасета и библиотек для работы

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
from dotenv import load_dotenv
import plotly.express as px
import os
load_dotenv()
api_key = os.getenv("API_KEY")
drive.mount('/content/drive')
data = pd.read_excel("/content/drive/MyDrive/RAD/NetflixShows.xlsx")
del data['ratingDescription']
data


### Обработка дубликатов

#### Удаление полных дубликатов

In [ ]:
print("Дубликатов:", data.duplicated().sum())
print("Доля:", data.duplicated().mean())
print("Среднее количество повторений:", len(data) / len(data.drop_duplicates()))


Полностью дублирующихся строк 500 - их можно удалить, не опасаясь потери информации. Доля дубликатов составляет 50% от общего объема и среднее количество повторений равное 2-м указывает на сильное дублирование информации. Это возможно при некорректной загрузке данных, ошибка могла произойти либо при объеденении данных, либо выгрузка была произведена дважды.
Поскольку 500 строк являются полностью идентичными - их можно удалить, не опасаясь потери информации.

In [ ]:
# Сгруппируем дубликаты по возрастным рейтингам для промотра того, какоц рейтинг имеет наибольшее количество дубликатов
# Источник: https://thetahat.ru/courses/python/08$0
pd.crosstab(data['rating'],
            data.duplicated(),
            colnames=None,
            margins=True
            ).set_axis(['unique', 'duplicates', 'all'],
                       axis=1
                       )


NameError: name 'pd' is not defined

При просмотре дубликатов по группам больше всего наблюдается TV-14, PG, G

In [ ]:
# Удалим полные дубликаты
data_no_duplicates = data.drop_duplicates()
data_no_duplicates.info()


#### Рассмотр дубликатов по title

Далее рассмотрим дубликаты по признаку title

In [ ]:
data_no_duplicates[data_no_duplicates.duplicated('title', keep=False)].sort_values(by='title')


Объекты отличаются по признакам rating и/или release year, а также по оценкам, что означает, что это разные шоу и их нужно оставить.

In [ ]:
df = data_no_duplicates.reset_index()


### Обработка пустых значений

Далее необходимо обработать пустые значения по каждому признаку

In [ ]:
df.isna().sum() / df.shape[0]


Видим пропуски в ratingLevel и user rating score.
1. Чтобы заполнить пропуски `ratingLevel` изучим его содержимое и позже заполним модой по соответствующему rating, так как данные признаки связаны: rating - сертификат контент рейтинга, ratingLevel - описание данного возрастного рейтинга и сцен присутствующих в фильме.
2. Пропуск по `user rating score` не получится обработать средствами данного датафрейма, так как пропуск значимый(>0.3)

In [ ]:
df_rating_nan = df.copy()
df_rating_nan['urs_nan'] = df_rating_nan['user rating score'].isnull()
df_rating_nan.groupby('rating')['urs_nan'].agg(nan=('sum'),
                                               all=('count'),
                                               percent=(lambda x: f'{x.sum() * 100 / x.count():.2f}%')
                                               )


Значения пропусков в `user rating score` в разных рейтинговых группах отличаюстя. Но тем не менее, в каждой рейтнговой группе доля пропусков является значимой и их заполнение внесет искажение в будущий анализ

#### Работа с рейтинговыми группами

In [ ]:
print(f'количество уникальных рейтинговых групп: {df['rating'].nunique()}')
print(f'список уникальных рейтинговых групп: {', '.join(df['rating'].unique())}')


Проверим зависимость между рейтинговыми группами и описанием группы, есть ли в них отличия или соответствие идет 1 к 1 - для одной группы одно описание.

In [ ]:
df.groupby('rating')['ratingLevel'].agg(num_unique='nunique', all='count')


на основе сводной таблицы выше можно сделать вывод, что большинство описаний имеют соответсвие 1 к 1. Но в группах PG, PG-13, R описаний много на одну группу, обработаем данные описания. Также в группе TV-14 есть два разных описания. Начнем с TV-14


In [ ]:
df[df['rating'] == 'TV-14'].groupby(['rating', 'ratingLevel']).size()


In [ ]:
df[df['ratingLevel'] == 'dialogue, language, sexual situations and violence']


Картина Hawaii Five-0 оценена рейтингом TV-14 и имеет отличное описание от всех остальных, в ней описываются сцены, из-за которых рейтинг оказался таким. Разделим описание на два вида. Одно с описанием рейтинга, второе с описанием сцен в фильме.

In [ ]:
df['sceneDesc'] = np.nan
df['sceneDesc'] = df['sceneDesc'].astype('object')
# Насколько безопасно менять значение по индексу?
df.loc[df['title'] == 'Hawaii Five-0', 'sceneDesc'] = df.loc[233, 'ratingLevel']
df.loc[df['title'] == 'Hawaii Five-0', 'ratingLevel'] = 'Parents strongly cautioned. May be unsuitable for children ages 14 and under.'
df[df['title'] == 'Hawaii Five-0']


In [ ]:
df[df['rating'] == 'TV-14'].groupby(['rating', 'ratingLevel']).size()

In [ ]:
df[df['rating'] == 'R'].groupby(['rating', 'ratingLevel']).size()

В рейтинге R всего 14 записей и все они разные. Но есть одна запись с описанием рейтинга, запишем в ratingLevel данную запись, а описаание сцен перенесем в соответствующий признак.

In [ ]:
df.loc[(df['rating'] == 'R') & (df['ratingLevel'] != 'Restricted. May be inappropriate for children 17 and under.'), 'sceneDesc'] = \
df[(df['rating'] == 'R') & (df['ratingLevel'] != 'Restricted. May be inappropriate for children 17 and under.')]['ratingLevel']

df.loc[(df['rating'] == 'R') & (df['ratingLevel'] != 'Restricted. May be inappropriate for children 17 and under.'), 'ratingLevel'] = \
'Restricted. May be inappropriate for children 17 and under.'
df[df['rating'] == 'R'].sample(1)

Даллее обработаем рейтинг PG-13

In [ ]:
df.loc[(df['rating'] == 'PG-13') & (df['ratingLevel'] != 'Parents strongly cautioned. May be inappropriate for children under 13.'), 'sceneDesc'] = \
df[(df['rating'] == 'PG-13') & (df['ratingLevel'] != 'Parents strongly cautioned. May be inappropriate for children under 13.')]['ratingLevel']

df.loc[(df['rating'] == 'PG-13') & (df['ratingLevel'] != 'Parents strongly cautioned. May be inappropriate for children under 13.'), 'ratingLevel'] = \
'Parents strongly cautioned. May be inappropriate for children under 13.'
df[df['rating'] == 'PG-13'].sample(1)

Далее преобразуем самый обширный рейтинг PG

In [ ]:
df[(df['rating'] == 'PG') & (df['ratingLevel'].str.contains('parent', case=False))].head()

В PG также есть описание рейтинга, проделаем тоже самое, что и в предыдущих шагах

In [ ]:
df.loc[(df['rating'] == 'PG') & (df['ratingLevel'] != 'Parental guidance suggested. May not be suitable for children.'), 'sceneDesc'] = \
df[(df['rating'] == 'PG') & (df['ratingLevel'] != 'Parental guidance suggested. May not be suitable for children.')]['ratingLevel']

df.loc[(df['rating'] == 'PG') & (df['ratingLevel'] != 'Parental guidance suggested. May not be suitable for children.'), 'ratingLevel'] = \
'Parental guidance suggested. May not be suitable for children.'

df[df['rating'] == 'PG'].sample(1)

После приведения ratingLevel к одному виду можем заполнить пустые значения, значениями из соответсвующего рейтинга

In [ ]:
moda = df.groupby('rating')['ratingLevel'].transform(lambda x: x.mode()[0])
df['ratingLevel'] = df['ratingLevel'].fillna(moda)

df.info()

### Feature engineering

#### Разделение объектов по типам на основе возрастного рейтнга

После изучения рейтинговых групп, группы, которые начинаются с TV являются телешоу (телесериалами) и на основе этих данных можно вывести информацию о типе объекта. Рейтинги TV-14, TV-PG, TV-MA, TV-Y, TV-Y7-FV, TV-G, TV-Y7 относятся к группе TV. Рейтинги: PG-13, R, PG, G к movie. Для них определим типы tv и movie. Ретинги NR и UR ставятся если фильм или тв сериал не оценен. Для них поставим тип unknown.

In [ ]:
df['type'] = np.nan
df['type'] = df['type'].astype('object')
df['type'] = np.where(df['rating'].isin(['TV-14', 'TV-PG', 'TV-MA', 'TV-Y', 'TV-Y7-FV', 'TV-G', 'TV-Y7']), 'tv',
                      np.where(df['rating'].isin(['PG-13', 'R', 'PG', 'G']), 'movie', 'unknown'))
df.sample(1)


#### Разделение на возрастные группы по возрастному рейтингу

Далее также попробуем выделить возрастные группы для которых предназначены шоу, поделим их на три:
1. детские: G, TV-Y, TV-Y7, TV-G, TV-Y7-FV
2. семейные: PG, TV-PG
3. подростковые: PG-13, TV-14
4. взрослые: TV-MA, R
5. Без рейтинга: NR, UR

In [ ]:
rating_to_segment = {
    'G': 'Kids',
    'TV-G': 'Kids',
    'TV-Y': 'Kids',
    'TV-Y7': 'Kids',
    'TV-Y7-FV': 'Kids',
    'PG': 'Family',
    'TV-PG': 'Family',
    'PG-13': 'Teens',
    'TV-14': 'Teens',
    'R': 'Adults',
    'TV-MA': 'Adults',
    'NR': 'Unrated',
    'UR': 'Unrated'
}

df = df.copy()
df['audience_segment'] = df['rating'].map(rating_to_segment).fillna('Other')

df[['title', 'rating', 'audience_segment']].sample()

#### Коэфициент привлекательности фильмов на основе возрастного рейтига

In [ ]:
users = df.pivot_table(
                    index='rating',
                    values='user rating size',
                    aggfunc='sum').reset_index()

users['mass_appeal'] = round((users['user rating size'] / users['user rating size'].max()),1)

df = pd.merge(df,users[['rating', 'mass_appeal']],
            on='rating',
            how='left')


def build_content_dna(row):

    rating = str(row['rating']).upper()

    # KIDS
    if rating in ['G', 'TV-G', 'TV-Y', 'TV-Y7', 'TV-Y7-FV']:
        kids = 1.0
    elif rating in ['PG', 'TV-PG']:
        kids = 0.7
    elif rating in ['PG-13', 'TV-14']:
        kids = 0.2
    else:
        kids = 0.0

       # YOUTH
    if rating in ['PG-13', 'TV-14']:
        youth = 1.0
    elif rating in ['PG', 'TV-PG']:
        youth = 0.7
    elif rating in ['R', 'TV-MA']:
        youth = 0.4
    else:
        youth = 0.2

    # ADULT
    if rating in ['R', 'TV-MA', 'NR', 'UR']:
        adult = 1.0

    elif rating in ['PG-13', 'TV-14']:
        adult = 0.6

    elif rating in ['PG', 'TV-PG']:
        adult = 0.4

    else:
        adult = 0.2

    return pd.Series({'kids_appeal': kids, 'youth_appeal': youth,'adult_appeal': adult})

# apply DNA mapping
df[['kids_appeal','youth_appeal','adult_appeal']] = df.apply(build_content_dna, axis=1)

content_dna = df[[
    'title',
    'rating',

    'user rating size',
    'age_group',
    'audience',
    'kids_appeal',
    'youth_appeal',
    'adult_appeal',
    'mass_appeal']]

content_dna.head(2)

NameError: name 'df' is not defined

### Обогащение данными из внешних источников

#### Поиск фильмов с помощью API OMDB

In [ ]:
import requests

def search_movie(df, title="title", year="release year", key=api_key):
  """
  Парметры:
  movie - строка с информацией о фильме
  title - название столбца с названиями фильмов
  year - название столбца с годом выпуска
  Вывод:
  жанр, страна, рейтинг IMDB, количество оценивших, возрастная группа

  Выполняет поиск фильма по названию и году через OMDB API
  """

  url = "http://www.omdbapi.com/"
  params = {
      "apikey": key,
      "t": df[title],
      "type": "movie",
      "y": df[year]
  }
  try:
    m = requests.get(url, params=params).json()
    return {'genre': m.get("Genre"),
            'country': m.get("Country"),
            'imdbRating': m.get("imdbRating"),
            'imdbVotes': m.get("imdbVotes"),
            'rated': m.get("Rated")
    }
  except Exception as e:
    print(e)

def parsing_OMDB(df):
  df['all_OMDB'] = np.nan
  # парсинг данных
  df['all_OMDB'] = df[df['type'] == 'movie'].apply(search_movie, axis=1)
  # разделение списка на отдельные признаки
  new_columns = pd.json_normalize(df['all_OMDB'])
  df = pd.concat([df, new_columns], axis=1)
  # Дополнительная проверка на соответствие
  mask = (
      (df['type'] == 'movie') &
      (df['rated'] != df['rating']) &
      (df['rated'].notna())
  )
  cols = ['genre', 'country', 'imdbRating', 'imdbVotes', 'rated']
  df.loc[mask, cols] = None
  df.drop(columns=['all_OMDB'], inplace=True)

def save_to_csv(df, filename='NetflixShows_clear.csv'):
    df.to_csv(filename, index=False)


NameError: name 'api_key' is not defined

In [ ]:
# Просмотр пустых значений
df[df['type'] == 'movie'].isna().sum() / df[df['type'] == 'movie'].shape[0]

Данный поиск проводится только по типу фильм, так как год выпуска телешоу в датасете указан неоднозначно, а для фильма год выпуска является точным значением. Парсинг проводится на основе `title, release year`, после проводится контрольная проверка по `rating`. При несоответсвии по одному из признаков информация удаляется.
Процент пропусков составил 13,5%, не найдено: 21 фильм.

## Визуализация и анализ

Целью нашего анализа является определение успешности шоу на основе его признаков, а также оценка успешности в разных странах на основе демографии.   
Задачи:
1. Анализ оценок фильмов и сериалов. Возможно ли с помощью user rating size и rating imdb оценить успешность тв-шоу? Мария
2. Исследование демографии стран, выявление зависимости между данным датасетом и демографией. Возможно ли на основе признаков датасета сделать вывод об об успешности шоу у разных возрастных групп? Светлана
3. Анализ данных о фильмах. Достаточно ли информации о фильмах, чтобы делать вывод об их успешности? Данил
4. Анализ данных о тв-шоу. Достаточно ли информации о фильмах, чтобы делать вывод об их успешности? Даниил
5. Описание фильма на основе иселдований. Дмитрий

### Мария. Анализ оценок

### Светлана. Демографический анализ

### Данил. Анализ фильмов

### Даниил. Анализ ТВ-шоу

**Цель этапа:** Исследовать сегмент сериалов на платформе Netflix, изучить динамику их выпуска, распределение по возрастным категориям и реакцию аудитории.

**Методология и подготовка данных:**
Из исходного массива выделяется подмножество телевизионного контента (`type == 'tv'`). Строки с пропущенными значениями в целевой переменной `user rating score` исключаются из анализа методом полного удаления (`dropna`). Искусственное заполнение пропусков (включая импутацию медианными или средними значениями) не применяется для сохранения исходного распределения оценок. Итоговый объем очищенной выборки для Блока 3 составляет 171 наблюдение.

В этом блоке мы:
1. Выделим телевизионный контент в отдельный датасет и очистим его от пропусков.
2. Изучим временную динамику производства и объясним аномальный всплеск 2016 года.
3. Проанализируем объём контента и оценки пользователей в разрезе возрастных рейтингов.

In [ ]:
# Еще раз грузим данные
df_raw = pd.read_csv("NetflixShows.csv")

# Фильтруем только ТВ-шоу и сразу удаляем строки, где нет оценок (user rating score)
df_tv = df_raw[df_raw['type'] == 'tv'].dropna(subset=['user rating score'])

# Оставляем только нужные для анализа колонки
columns_to_keep = ['title', 'rating', 'release year', 'user rating score']
df_tv = df_tv[columns_to_keep]

# Посмотрим на размер очищенного датасета и первые несколько строк
print(f"Размер очищенного датасета по ТВ-шоу: {df_tv.shape}")
df_tv.head()

#### Динамика выпуска ТВ-шоу по годам

Агрегируем данные, чтобы посмотреть, сколько сериалов из нашего датасета выпускалось в разные годы, и рассчитаем их средний рейтинг.

In [ ]:
# Группируем по годам и считаем количество сериалов и среднюю оценку
yearly_stats = df_tv.groupby('release year')['user rating score'].agg(['count', 'mean']).reset_index()
yearly_stats.columns = ['Год выпуска', 'Количество шоу', 'Средний рейтинг']

# Округляем рейтинг
yearly_stats['Средний рейтинг'] = yearly_stats['Средний рейтинг'].round(2)

print("Динамика по годам:")
display(yearly_stats.sort_values(by='Год выпуска', ascending=False))

**Промежуточный вывод:**

Анализ временной динамики фиксирует аномальный скачок объема контента в **2016 году**: количество сериалов в выборке достигло 68 единиц, что составляет **39.7% от всей исследуемой выборки ТВ-шоу** и демонстрирует рост почти в 3 раза относительно 2015 года (25 единиц).

Этот всплеск в датасете совпадает с подтвержденными рыночными данными о глобальной экспансии Netflix в январе 2016 года на 190 стран, что потребовало экстренного наращивания библиотеки для удержания новых рынков. При этом средний рейтинг удерживается на стабильном уровне (82.81 балла), что указывает на сохранение контроля качества при кратном увеличении объема производства. В 2017 году наблюдается рост среднего рейтинга до пиковых 88.13 балла при объеме в 16 шоу.

#### Распределение объёма контента по сегментам аудитории

Посмотрим, на какую именно аудиторию ориентированы сериалы Netflix, подсчитав количество проектов в разрезе укрупненных сегментов (`audience_segment`), полученных на основе исходных возрастных рейтингов.

In [ ]:
# Создаем словарь для маппинга возрастных рейтингов в укрупненные сегменты
rating_to_segment = {
    'G': 'Kids', 'TV-G': 'Kids', 'TV-Y': 'Kids', 'TV-Y7': 'Kids', 'TV-Y7-FV': 'Kids',
    'PG': 'Family', 'TV-PG': 'Family',
    'PG-13': 'Teens', 'TV-14': 'Teens',
    'R': 'Adults', 'TV-MA': 'Adults',
    'NR': 'Unrated', 'UR': 'Unrated'
}

# Применяем маппинг к нашему датасету ТВ-шоу
df_tv = df_tv.copy() # Обработаем предупреждение SettingWithCopyWarning
df_tv['audience_segment'] = df_tv['rating'].map(rating_to_segment).fillna('Other')

# Считаем количество шоу по укрупненным сегментам аудитории
segment_counts = df_tv['audience_segment'].value_counts().reset_index()

# Переименовываем колонки для наглядности
segment_counts.columns = ['Сегмент аудитории', 'Количество сериалов']

display(segment_counts)

In [ ]:
# Строим простой график с помощью модуля express библиотеки plotly (https://plotly.com/python/bar-charts/)
fig_volume = px.bar(segment_counts,
                    x='Сегмент аудитории',
                    y='Количество сериалов',
                    title='Объем выпущенных ТВ-шоу по сегментам аудитории',
                    color='Сегмент аудитории',
                    template='plotly_dark')
fig_volume.show()

**Промежуточный вывод:**

Визуализация распределения ТВ-шоу по сегментам аудитории позволяет сделать следующие выводы о контентной стратегии платформы:

1. **Доминирующий сегмент (Ядро аудитории):** Абсолютным лидером по объему контента является категория `Teens` (**77 шоу**), за которой следует `Adults` (**40 шоу**). Суммарно эти два сегмента занимают **68.4% всего анализируемого контента**. Это доказывает, что Netflix делает ключевую ставку на подростков старшего возраста и взрослую аудиторию. Именно эти группы формируют наиболее активное и платежеспособное ядро подписчиков.
2. **Детский контент:** Сегмент `Kids` занимает третье место (**33 шоу**), агрегируя в себе множество мелких индивидуальных детских рейтингов.
3. **Семейный контент:** Категория `Family` замыкает распределение (**21 шоу**), выступая в роли безопасного компромисса для совместного просмотра.

Такая структура подтверждает, что в сегменте сериалов Netflix сохраняет фокус на более зрелом, драматическом и сложносюжетном контенте, производя детские и семейные проекты как сопутствующие.

#### Анализ оценок пользователей по сегментам аудитории

Теперь проверим, контент для каких укрупненных сегментов аудитории получает самые высокие оценки от пользователей.

In [ ]:
# Явный расчет медианных оценок по укрупненным сегментам ########## Не знаю нужно это или просто Box Plot, который ниже, но пусть будет для наглядности

median_segment_ratings = df_tv.groupby('audience_segment')['user rating score'].median().sort_values(ascending=False).reset_index()
median_segment_ratings.columns = ['Сегмент аудитории', 'Медианная оценка']
print("Медианные оценки по сегментам аудитории:")
display(median_segment_ratings)

# Строим Box Plot по укрупненным сегментам аудитории (https://plotly.com/python/bar-charts/)
fig_ratings = px.box(df_tv,
                     x='audience_segment',
                     y='user rating score',
                     color='audience_segment',
                     title='Распределение оценок ТВ-шоу по сегментам аудитории',
                     labels={'user rating score': 'Оценка пользователей', 'audience_segment': 'Сегмент аудитории'},
                     template='plotly_dark')
fig_ratings.show()

**Промежуточный вывод:**

Анализ распределения оценок (`user rating score`) с помощью графика Box Plot и расчет медианных значений выявили четкую зависимость лояльности аудитории от целевого сегмента контента:

1. **Высокая лояльность к зрелому и семейному контенту:** Самую высокую медианную оценку демонстрирует категория `Adults` (**89.0 баллов**). Практически на том же уровне находится семейный контент `Family` (**88.0 баллов**), а за ними с минимальным отрывом идет подростковый сегмент `Teens` (**86.0 баллов**). Это доказывает, что взрослая, семейная и подростковая аудитория Netflix максимально вовлечена, а платформа успешно создает качественные шоу, точно попадающие в запросы своего ключевого ядра.
2. **Проседание детского сегмента:** Сегмент `Kids` показывает заметный спад удовлетворенности — его медианная оценка составляет всего **74.0 балла**. Это подтверждает, что удержание детской аудитории не является главным приоритетом платформы в рамках данного формата.

*Техническое замечание:* Все расчеты и метрики в данном блоке получены строго на основе исходных заполненных значений (`dropna()`). Искусственное заполнение пропущенных оценок медианой по подвыборкам не применялось, чтобы исключить искажение реальной картины распределения.

### Общий вывод по Блоку 3 (Корреляционный анализ и оценка достаточности данных)

1. **Характер появления контента и взаимосвязь метрик:**
   В сегменте ТВ-шоу прослеживается жесткая производственная централизация: платформой зафиксирован фокус на сегментах `Teens` (77 шоу) и `Adults` (40 шоу), занимающих **68.4% всего объема**. Наблюдается прямая положительная корреляция между объемом контента и его успешностью: наиболее массовые категории получают наивысший отклик и лояльность пользователей (медианы 86.0 и 89.0 баллов соответственно). Временной фактор (всплеск 2016 года) подтверждает, что масштабирование библиотеки происходило именно за счет этих целевых групп, где количество совпало с качеством. Детский контент (`Kids`) производится в меньших объемах и имеет более низкие метрики одобрения (медиана 74.0 балла). Все расчеты проведены строго на очищенных данных (171 строка) без применения методов искусственного заполнения пропусков.

2. **Оценка достаточности информации для прогнозирования успешности контента:**
   Текущего набора признаков (объем выборки, сегменты аудитории и агрегированная оценка `user rating score`) **недостаточно** для построения объективной прогностической модели успешности ТВ-шоу.
   
   * Мы видим только результирующую оценку, но не имеем доступа к ключевым предикторам: жанровой структуре, бюджетам серий, касту актеров, количеству сезонов и объему просмотров в часах.
   * Данное ограничение требует объединения нашей аналитики по сериалам с результатами блоков по полнометражным фильмам (Movies). Сравнение внутренней структуры этих двух макро-сегментов позволит всей команде сформировать комплексную итоговую гипотезу о факторах, предопределяющих успех контента на платформе.

### Дмитрий. Вывод